# 03b_segment_segger: run segger and convert its output, from Python / Jupyter.

```text
Python version of 03b_segment_segger.sh: same three steps, same outputs.
  1. segger segment            transcript -> cell assignment (needs a CUDA GPU)
  2. segger export boundaries  cell polygons -> cell areas (optional)
  3. conversion to the common format read by R/helpers.R (03b_segger_to_common.py)

segger lives in its own pixi environment (Python 3.11 + CUDA + RAPIDS), so steps
1-2 are run as `pixi run -e cuda121 segger ...` from here, while this script /
notebook itself runs in exp1-imaging. For step 1 the notebook must run on a GPU
node (e.g. Jupyter started inside a GPU job). Steps 2-3 also work without GPU:
re-run with skip_segment=True to redo only those.

One-time install of segger (the login node is fine; it does not need the GPU):
  curl -fsSL https://pixi.sh/install.sh | sh  &&  source ~/.bashrc
  git clone https://github.com/dpeerlab/segger.git /work/.../segger
  cd /work/.../segger  &&  CONDA_OVERRIDE_CUDA=12.1 pixi install -e cuda121

Outputs
  <project>/data/segmentation/segger_raw/<slide_id>/segger_segmentation.parquet (+ export/)
  <project>/data/segmentation/segger/<slide_id>/   common format for 03c / load_segmentation()

Example
  python 03b_segment_segger.py --slide-id TMA_ORGAN --xenium-dir /path/to/output-XETG... \
      --project-dir /work/Spatial_TMA --segger-repo /work/segger
```

**Notebook version of `experiment1_tma/03b_segment_segger.py`.** Same code, laid out for interactive work:

1. Set the options in the **Parameters** cell (the script's command-line options: `--slide-id` becomes `slide_id`).
2. Run the cells in order. After each cell you can inspect its variables, tables and images.

Use the `.py` script for batch jobs on the cluster; both produce the same outputs. This notebook is generated by `tools/make_notebooks.py`, which overwrites it, so save your own edits under another name.

In [ ]:
import os

# folder of 03b_segment_segger.py; this notebook sits in notebooks/ next to it
SCRIPT_DIR = os.path.abspath('..')

# Which Python runs this notebook? It must be the conda environment, not the system one.
import sys
print('Python:', sys.executable)
try:
    import numpy  # noqa: F401
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'This kernel is not the exp1-imaging environment. In a terminal, once:\n'
        '  conda activate exp1-imaging\n'
        '  python -m ipykernel install --user --name exp1-imaging --display-name exp1-imaging\n'
        'then in Jupyter: Kernel > Change kernel > exp1-imaging  (environment file: experiment1_tma/env/py_imaging.yml)')

# Packages this step needs (import name: pip name). Anything missing is installed INTO THIS
# kernel with %pip in a notebook cell -- not with pip in a terminal, which may be another Python.
import importlib.util
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'pyarrow': 'pyarrow'}
missing = [pip for mod, pip in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError(
        f'Missing in this kernel: {", ".join(missing)}\n'
        f'Run this in a new cell:  %pip install {" ".join(missing)}\n'
        'then Kernel > Restart, and run the notebook again from the top.')
print('All packages found.')

In [ ]:
import argparse
import os
import shlex
import shutil
import subprocess
import sys
import time
from importlib import import_module

HERE = SCRIPT_DIR
sys.path.insert(0, HERE)

## Parameters

In [ ]:
from types import SimpleNamespace

# Same options as `python experiment1_tma/03b_segment_segger.py --help` (--slide-id -> slide_id).

# Options marked REQUIRED have no default: replace their None, e.g. slide_id="TMA_ORGAN".
# Paths are plain strings in quotes.

a = SimpleNamespace(
    slide_id=None,           # REQUIRED
    xenium_dir=None,         # REQUIRED. Xenium output folder (output-XETG...) of this slide
    project_dir=None,        # REQUIRED. project root (EXP1_PROJECT_DIR)
    segger_repo="~/segger",  # folder of the segger git clone (with pixi.toml)
    pixi=None,               # path to the pixi program; default: on PATH or ~/.pixi/bin/pixi
    pixi_env="cuda121",      # pixi environment of segger
    segger_options="",       # extra options for `segger segment` as one string, e.g. '--n-epochs 30' (command line: --segger-options='--n-epochs 30'; list: pixi run -e cuda121 segger segment --help)
    skip_segment=False,      # reuse an existing segger_segmentation.parquet; only redo export + conversion
    skip_export=False,       # no boundary export (cell areas stay NA)
    cores_csv=None,          # default: <project>/tables/02_cores_all.csv
    min_transcripts=1,
    min_qv=20.0,
)

missing = [k for k in ('slide_id', 'xenium_dir', 'project_dir') if getattr(a, k) in (None, '')]
if missing:
    raise ValueError(f'Fill in these options in the cell above (replace None with a value): {missing}')

## Helper functions

In [ ]:
def find_pixi(path=None):
    """pixi is often not on Jupyter's PATH (the installer only edits ~/.bashrc)."""
    for c in (path, shutil.which("pixi"), os.path.expanduser("~/.pixi/bin/pixi")):
        if c and os.path.isfile(os.path.expanduser(c)) and os.access(os.path.expanduser(c), os.X_OK):
            return os.path.expanduser(c)
    raise FileNotFoundError("pixi not found. Install it (curl -fsSL https://pixi.sh/install.sh | sh) "
                            "or give its full path as --pixi / pixi=...")

In [ ]:
def run(cmd, cwd):
    """Run a command and stream its output line by line (terminal and Jupyter). Returns the exit code."""
    print("$", " ".join(cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"[exit code {rc}, {(time.time() - t0) / 60:.1f} min]", flush=True)
    return rc

In [ ]:
def gpu_report():
    smi = shutil.which("nvidia-smi")
    if smi is None:
        print("WARNING: nvidia-smi not found -- this does not look like a GPU node, and segger segment "
              "needs a CUDA GPU. Start Jupyter (or this script) inside a GPU job.")
        return False
    out = subprocess.run([smi, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                         capture_output=True, text=True)
    print("GPU:", out.stdout.strip() or out.stderr.strip())
    return out.returncode == 0

## Run

In [ ]:
repo = os.path.expanduser(a.segger_repo)
raw_out = os.path.join(a.project_dir, "data", "segmentation", "segger_raw", a.slide_id)
export_dir = os.path.join(raw_out, "export")
common_out = os.path.join(a.project_dir, "data", "segmentation", "segger", a.slide_id)
cores_csv = a.cores_csv or os.path.join(a.project_dir, "tables", "02_cores_all.csv")
seg_parquet = os.path.join(raw_out, "segger_segmentation.parquet")
for path, what in ((a.xenium_dir, "Xenium folder"), (os.path.join(repo, "pixi.toml"), "segger clone"),
                   (cores_csv, "core table from 02")):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{what} not found: {path}")
for d in (raw_out, export_dir, common_out):
    os.makedirs(d, exist_ok=True)
pixi = find_pixi(a.pixi)
print(f"pixi: {pixi}\nsegger: {repo} (env {a.pixi_env})\noutput: {raw_out}")

### 1. segger segment (GPU)

In [ ]:
if a.skip_segment:
    print(f"skip_segment: reusing {seg_parquet}")
else:
    gpu_report()
    rc = run([pixi, "run", "-e", a.pixi_env, "segger", "segment", "-i", a.xenium_dir, "-o", raw_out,
              *shlex.split(a.segger_options)], cwd=repo)
    if rc != 0:
        raise RuntimeError("segger segment failed -- see its output above")
if not os.path.isfile(seg_parquet):
    raise FileNotFoundError(f"{seg_parquet} was not written. Files in {raw_out}: {sorted(os.listdir(raw_out))}")

### 2. segger export boundaries (cell polygons -> cell areas)

In [ ]:
if not a.skip_export:
    rc = run([pixi, "run", "-e", a.pixi_env, "segger", "export", "boundaries",
              "-s", seg_parquet, "-o", export_dir], cwd=repo)
    if rc != 0:
        print("boundary export failed -- continuing without cell areas")
boundaries = os.path.join(export_dir, "cell_boundaries.parquet")
if not os.path.isfile(boundaries):
    boundaries = None

### 3. common format (same code as 03b_segger_to_common.py)

In [ ]:
conv = import_module("03b_segger_to_common")
argv = ["--slide-id", a.slide_id, "--xenium-dir", a.xenium_dir, "--segger-parquet", seg_parquet,
        "--cores-csv", cores_csv, "--out-dir", common_out,
        "--min-transcripts", str(a.min_transcripts), "--min-qv", str(a.min_qv)]
if boundaries:
    argv += ["--boundaries", boundaries]
conv.main(argv)
print(f"Done: {common_out}\nNext: the other slide, then 03c_segmentation_benchmark.R")